# B2-023-generative-models-diffusion — Practice p09 — Solution

**Type:** constrained-coding · **Difficulty:** intro · **Concepts:** denoising-diffusion-probabilistic-models

*50 minutes.*  
**Set:** B  
**Program layer:** Round 2 extension  
**Compute:** `compute.policy: cpu` · seed `20261015`  
**Qualified prerequisites:** `book1:F1-scientific-python`, `book1:F3-matrices`, `book1:F4-multivar-calculus`, `book1:F5-probability`, `book1:C1-ml-fundamentals`, `book1:C2-linear-models`, `book1:C5-neural-networks`, `book1:C6-pytorch`, `book1:C11-neural-training`, `B2-019-attention-transformers`, `B2-020-language-transformers`, `B2-021-cross-modal-transformers-vision`, `B2-022-probabilistic-latent-models`  
**Remediation links actually used:** [book1:F1-scientific-python](../../../../book1/units/F1-scientific-python/lesson.ipynb), [book1:F3-matrices](../../../../book1/units/F3-matrices/lesson.ipynb), [book1:F4-multivar-calculus](../../../../book1/units/F4-multivar-calculus/lesson.ipynb), [book1:F5-probability](../../../../book1/units/F5-probability/lesson.ipynb), [book1:C1-ml-fundamentals](../../../../book1/units/C1-ml-fundamentals/lesson.ipynb), [book1:C2-linear-models](../../../../book1/units/C2-linear-models/lesson.ipynb), [book1:C5-neural-networks](../../../../book1/units/C5-neural-networks/lesson.ipynb), [book1:C6-pytorch](../../../../book1/units/C6-pytorch/lesson.ipynb), [book1:C11-neural-training](../../../../book1/units/C11-neural-training/lesson.ipynb), [B2-019-attention-transformers](../../B2-019-attention-transformers/lesson.ipynb), [B2-020-language-transformers](../../B2-020-language-transformers/lesson.ipynb), [B2-021-cross-modal-transformers-vision](../../B2-021-cross-modal-transformers-vision/lesson.ipynb), [B2-022-probabilistic-latent-models](../../B2-022-probabilistic-latent-models/lesson.ipynb).

**Data boundary:** CPU only; use literal unit data, the seeded unit generator, and standard library/NumPy/Torch only. Do not use the web, downloads, external datasets, torchvision datasets, model hubs, pretrained models, checkpoints, or student data.

## Supplied cell (copied verbatim from the statement)

In [ ]:
import torch

SEED = 20261015
X0 = torch.tensor([[1.0, -2.0], [0.5, 0.5], [-1.0, 3.0]])
T_IDX = torch.tensor([1, 3, 5])
EPS = torch.tensor([[0.5, 0.0], [-1.0, 2.0], [0.2, -0.4]])
ATOL = 1e-12
RTOL = 1e-12

## Solution

`make_schedule` stores every table with the $(T+1)$ convention: index $t$ holds timestep $t$, and index $0$ is padding ($\beta_0=0$, $\alpha_0=1$, $\bar\alpha_0=1$, $\tilde\beta_0=0$).
`torch.linspace(beta_1, beta_T, T)` includes both endpoints, so `beta[1] == beta_1` and `beta[T] == beta_T`.
$\tilde\beta_t=\frac{1-\bar\alpha_{t-1}}{1-\bar\alpha_t}\beta_t$ is computed for all $t\ge1$ at once; at $t=1$ the numerator $1-\bar\alpha_0=0$ gives $\tilde\beta_1=0$.

`q_sample` gathers `alpha_bar[t]` (shape `(B,)`), casts it to `x0`'s dtype, unsqueezes it to `(B,1)` so that each row's coefficient broadcasts across that row's coordinates, and applies the closed form $x_t=\sqrt{\bar\alpha_t}\,x_0+\sqrt{1-\bar\alpha_t}\,\varepsilon$ with no loops and no random draws.

In [ ]:
def make_schedule(T, beta_1, beta_T):
    if not isinstance(T, int) or isinstance(T, bool) or T < 2:
        raise ValueError("T must be an int >= 2")
    if not (0 < beta_1 <= beta_T < 1):
        raise ValueError("need 0 < beta_1 <= beta_T < 1")
    beta = torch.zeros(T + 1, dtype=torch.float64)
    beta[1:] = torch.linspace(beta_1, beta_T, T, dtype=torch.float64)
    alpha = 1 - beta
    alpha_bar = torch.cumprod(alpha, dim=0)
    beta_tilde = torch.zeros(T + 1, dtype=torch.float64)
    beta_tilde[1:] = (1 - alpha_bar[:-1]) / (1 - alpha_bar[1:]) * beta[1:]
    return {"beta": beta, "alpha": alpha, "alpha_bar": alpha_bar, "beta_tilde": beta_tilde}


def q_sample(x0, t, eps, alpha_bar):
    if x0.shape != eps.shape:
        raise ValueError("x0 and eps must have the same shape")
    if (not isinstance(t, torch.Tensor) or t.dtype != torch.int64 or t.dim() != 1
            or x0.dim() != 2 or t.shape[0] != x0.shape[0]):
        raise ValueError("t must be an int64 tensor of shape (B,)")
    T = alpha_bar.shape[0] - 1
    if bool((t < 1).any()) or bool((t > T).any()):
        raise ValueError("timesteps must lie in 1..T")
    ab = alpha_bar[t].to(x0.dtype).unsqueeze(1)
    return ab.sqrt() * x0 + (1 - ab).sqrt() * eps

### Probes 1–3 and the two defective variants

In [ ]:
sched5 = make_schedule(5, 0.1, 0.5)
expected5 = {
    "beta": [0, 0.1, 0.2, 0.3, 0.4, 0.5],
    "alpha": [1, 0.9, 0.8, 0.7, 0.6, 0.5],
    "alpha_bar": [1, 0.9, 0.72, 0.504, 0.3024, 0.1512],
    "beta_tilde": [0, 0, 0.0714285714285714, 0.1693548387096774, 0.2844036697247707, 0.4109330819981150],
}
probe1 = {k: torch.allclose(sched5[k], torch.tensor(v, dtype=torch.float64), atol=ATOL, rtol=RTOL)
          for k, v in expected5.items()}

sched50 = make_schedule(50, 1e-3, 0.2)


def endpoint_probe(beta, T, beta_1, beta_T):
    return abs(beta[1].item() - beta_1) <= 1e-15 and abs(beta[T].item() - beta_T) <= 1e-15


off_by_one_beta = torch.zeros(51, dtype=torch.float64)
off_by_one_beta[1:] = torch.linspace(1e-3, 0.2, 51, dtype=torch.float64)[1:]

xt = q_sample(X0, T_IDX, EPS, sched5["alpha_bar"])
expected_xt = torch.tensor([[1.1067972183, -1.8973665237], [-0.3493078947, 1.7635102272],
                            [-0.2045837343, 0.7980118394]])


def q_sample_alpha_bug(x0, t, eps, alpha):
    a = alpha[t].to(x0.dtype).unsqueeze(1)
    return a.sqrt() * x0 + (1 - a).sqrt() * eps


xt_bug = q_sample_alpha_bug(X0, T_IDX, EPS, sched5["alpha"])
row_diff = (xt_bug - xt).abs().max(dim=1).values
print("probe 1:", probe1)
print("probe 2: alpha_bar[50] =", sched50["alpha_bar"][50].item(), " off-by-one beta[1] =", off_by_one_beta[1].item())
print("probe 3:", xt.tolist(), " alpha-bug row diffs:", row_diff.tolist())

### Probe 4: statistical check of the closed form

With $200{,}000$ float64 draws the standard error of each mean is about $\sqrt{0.724/200000}\approx0.0019$, so the `0.01` tolerances are about five standard errors wide.

In [ ]:
N = 200_000
x0_stat = torch.tensor([1.0, -1.0], dtype=torch.float64).repeat(N, 1)
t_stat = torch.full((N,), 25, dtype=torch.int64)
eps_stat = torch.randn(N, 2, generator=torch.Generator().manual_seed(SEED), dtype=torch.float64)
x25 = q_sample(x0_stat, t_stat, eps_stat, sched50["alpha_bar"])
ab25 = sched50["alpha_bar"][25]
target_mean = ab25.sqrt() * x0_stat[0]
print("alpha_bar[25] =", ab25.item(), " mean", x25.mean(dim=0).tolist(), " var", x25.var(dim=0).tolist())

### Probe 5: invalid inputs

In [ ]:
def raises_value_error(fn):
    try:
        fn()
    except ValueError:
        return True
    return False


invalid_calls = [
    lambda: make_schedule(1, 0.1, 0.5),                  # T < 2
    lambda: make_schedule(5.0, 0.1, 0.5),                # T not an int
    lambda: make_schedule(5, 0.0, 0.5),                  # beta_1 <= 0
    lambda: make_schedule(5, 0.6, 0.5),                  # beta_1 > beta_T
    lambda: make_schedule(5, 0.1, 1.0),                  # beta_T >= 1
    lambda: q_sample(X0, T_IDX, EPS[:2], sched5["alpha_bar"]),                     # shapes differ
    lambda: q_sample(X0, T_IDX.to(torch.int32), EPS, sched5["alpha_bar"]),         # not int64
    lambda: q_sample(X0, T_IDX[:, None], EPS, sched5["alpha_bar"]),                # not shape (B,)
    lambda: q_sample(X0, torch.tensor([0, 3, 5]), EPS, sched5["alpha_bar"]),       # t = 0
    lambda: q_sample(X0, torch.tensor([1, 3, 6]), EPS, sched5["alpha_bar"]),       # t > T
]
invalid_results = [raises_value_error(fn) for fn in invalid_calls]
print("probe 5:", invalid_results)

### Answer check

In [ ]:
# Probe 1
assert all(probe1.values()), probe1
# Probe 2
for key in ("beta", "alpha", "alpha_bar", "beta_tilde"):
    assert tuple(sched50[key].shape) == (51,) and sched50[key].dtype == torch.float64
assert endpoint_probe(sched50["beta"], 50, 1e-3, 0.2)
assert abs(sched50["alpha_bar"][50].item() - 0.004505985481139705) <= 1e-12
assert bool((sched50["alpha_bar"][1:][1:] < sched50["alpha_bar"][1:][:-1]).all())
assert not endpoint_probe(off_by_one_beta, 50, 1e-3, 0.2)
# Probe 3
assert xt.dtype == torch.float32 and xt.shape == X0.shape
assert torch.allclose(xt, expected_xt, atol=1e-6, rtol=1e-6)
assert torch.allclose(xt_bug[0], xt[0], atol=1e-6, rtol=1e-6)
assert row_diff[1].item() > 0.1 and row_diff[2].item() > 0.1
# Probe 4
assert abs(ab25.item() - 0.2760332057) < 1e-9
assert torch.allclose(x25.mean(dim=0), target_mean, atol=0.01, rtol=0.0)
assert torch.allclose(x25.var(dim=0), (1 - ab25).expand(2), atol=0.01, rtol=0.0)
# Probe 5
assert all(invalid_results), invalid_results